In [2]:
import heapq
 
DETAILED = False          # True  -> full step-by-step explanation
SHOW_ALL_ROUTES = False   # True  -> also list every route as a check

In [3]:
# Directed graph (state space): town -> {connected town: step cost in km}
# Step costs are the average distances from the Part 1 table.
GRAPH = {
    "Nairobi":   {"Naivasha": 89.5, "Narok": 141.5, "Thika": 44},
    "Naivasha":  {"Nakuru": 71},
    "Nakuru":    {"Eldoret": 156, "Kericho": 107},
    "Eldoret":   {"Kitale": 71},
    "Kitale":    {"Kakamega": 110},
    "Kakamega":  {"Kisumu": 50},
    "Kericho":   {"Sondu": 45, "Kisii": 87.2},
    "Sondu":     {"Kisumu": 52.6},
    "Narok":     {"Bomet": 81.5},
    "Bomet":     {"Kericho": 74.25},
    "Kisii":     {"Sondu": 61.3},
    "Thika":     {"Nyeri": 110},
    "Nyeri":     {"Nyahururu": 104},
    "Nyahururu": {"Nakuru": 64},
    "Kisumu":    {},                      # goal state: no further actions needed
}
 
INITIAL_STATE = "Nairobi"
GOAL_STATE = "Kisumu"
 
 
def goal_test(state):
    """Goal test: is the trailer in Kisumu?"""
    return state == GOAL_STATE

In [4]:
2. # UNIFORM COST SEARCH

def show_frontier(frontier, explored, best_cost):
    """Print the live frontier (nodes waiting to be expanded), cheapest first."""
    live = sorted(
        (cost, node) for cost, _, node, _ in frontier
        if node not in explored and cost == best_cost[node]
    )
    return ", ".join(f"{node} ({cost:.2f})" for cost, node in live)
 
 
def log(*args):
    """Print only in DETAILED mode."""
    if DETAILED:
        print(*args)
 
 
def uniform_cost_search(graph, start):
    """Return (optimal path, optimal cost). Prints the traversal as it goes."""
    counter = 0                                   # tie-breaker for equal costs
    frontier = [(0, counter, start, [start])]     # priority queue: (g, tie, node, path)
    best_cost = {start: 0}                        # cheapest known cost to each town
    explored = set()                              # towns already expanded
    iteration = 0
 
    print(f"Initial state: {start}    Goal state: {GOAL_STATE}\n")
    if not DETAILED:
        print(f"{'Iter':<5}{'Expand':<12}{'Cost (km)':>9}   Frontier after expansion")
 
    while frontier:                               # stop if the frontier is empty (failure)
        # Remove the lowest-cost node from the frontier
        cost, _, node, path = heapq.heappop(frontier)
        if node in explored:                      # stale, more expensive duplicate
            continue
        iteration += 1
 
        log(f"--- Iteration {iteration} ---")
        log(f"Expand   : {node}  | path cost g = {cost:.2f} km")
        log(f"Path     : {' -> '.join(path)}")
 
        # Goal test when the node is selected for expansion
        if goal_test(node):
            log(f"Goal test: {node} == {GOAL_STATE}?  YES -> solution found\n")
            if not DETAILED:
                print(f"{iteration:<5}{node:<12}{cost:>9.2f}   GOAL REACHED")
            return path, cost
        log(f"Goal test: {node} == {GOAL_STATE}?  No")
 
        explored.add(node)
 
        # Expand: apply every legal action (drive to each connected town)
        for neighbour, step_cost in graph[node].items():
            new_cost = cost + step_cost
            if neighbour in explored:
                log(f"  - {neighbour}: already expanded, ignored")
            elif new_cost < best_cost.get(neighbour, float("inf")):
                best_cost[neighbour] = new_cost
                counter += 1
                heapq.heappush(frontier, (new_cost, counter, neighbour, path + [neighbour]))
                log(f"  + {neighbour}: {cost:.2f} + {step_cost} = {new_cost:.2f}  (added to frontier)")
            else:
                log(f"  x {neighbour}: {new_cost:.2f} is not cheaper than "
                    f"{best_cost[neighbour]:.2f}, ignored")
 
        if DETAILED:
            print("Frontier :", show_frontier(frontier, explored, best_cost), "\n")
        else:
            print(f"{iteration:<5}{node:<12}{cost:>9.2f}   "
                  f"{show_frontier(frontier, explored, best_cost)}")
 
    return None, float("inf")   

In [5]:
# CHECK: list every route in the graph and its total cost

def all_routes(graph, node, goal, path=None, cost=0):
    """Depth-first listing of every simple route from node to goal."""
    path = (path or []) + [node]
    if node == goal:
        yield path, cost
        return
    for neighbour, step_cost in graph[node].items():
        if neighbour not in path:
            yield from all_routes(graph, neighbour, goal, path, cost + step_cost)

In [6]:

# RUN THE PROGRAM
if __name__ == "__main__":
    solution, total_cost = uniform_cost_search(GRAPH, INITIAL_STATE)
 
    print("=" * 60)
    print("RESULT")
    print("=" * 60)
    if solution:
        print("Optimal path :", " -> ".join(solution))
        print(f"Path cost    : {total_cost:.2f} km")
    else:
        print("No route found.")
 
    if SHOW_ALL_ROUTES:
        print("\nCheck - all routes in the graph, cheapest first:")
        for route, route_cost in sorted(all_routes(GRAPH, INITIAL_STATE, GOAL_STATE),
                                        key=lambda r: r[1]):
            print(f"  {route_cost:8.2f} km  {' -> '.join(route)}")
 

Initial state: Nairobi    Goal state: Kisumu

Iter Expand      Cost (km)   Frontier after expansion
1    Nairobi          0.00   Thika (44.00), Naivasha (89.50), Narok (141.50)
2    Thika           44.00   Naivasha (89.50), Narok (141.50), Nyeri (154.00)
3    Naivasha        89.50   Narok (141.50), Nyeri (154.00), Nakuru (160.50)
4    Narok          141.50   Nyeri (154.00), Nakuru (160.50), Bomet (223.00)
5    Nyeri          154.00   Nakuru (160.50), Bomet (223.00), Nyahururu (258.00)
6    Nakuru         160.50   Bomet (223.00), Nyahururu (258.00), Kericho (267.50), Eldoret (316.50)
7    Bomet          223.00   Nyahururu (258.00), Kericho (267.50), Eldoret (316.50)
8    Nyahururu      258.00   Kericho (267.50), Eldoret (316.50)
9    Kericho        267.50   Sondu (312.50), Eldoret (316.50), Kisii (354.70)
10   Sondu          312.50   Eldoret (316.50), Kisii (354.70), Kisumu (365.10)
11   Eldoret        316.50   Kisii (354.70), Kisumu (365.10), Kitale (387.50)
12   Kisii          354.70 